#### Using RAG (with sqlite search)

Our current RAG pipeline loads data and builds the in-memory `minsearch` index at startup, which runs efficiently within a single process for small FAQ datasets. However, this monolithic approach breaks down as datasets scale to millions of documents because data preparation—including API calls, parsing, and cleaning—becomes sluggish. Because `minsearch` is memory-bound, all indexed data vanishes when the process stops, causing wasteful re-indexing delays during service restarts.

To resolve this inefficiency, we must decouple the data ingestion process from the querying system into independent operations. Under this architecture, a dedicated data engineering ingestion process writes the source data once into a persistent search index, allowing separate querying processes to read from it continuously across restarts. While robust enterprise tools like Elasticsearch, OpenSearch, or Qdrant are excellent choices for a persistent backend, this module utilizes `sqlitesearch`. Built as a lightweight, drop-in Python wrapper around SQLite's native FTS5 full-text search capability, `sqlitesearch` provides full persistence without requiring any external software dependencies.

To use `sqlitesearch` run below command on the terminal:
```python
uv add sqlitesearch
```

In [1]:
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

In [2]:
sqlite_index.count()

79

In [3]:
results = sqlite_index.search("Can I still join the course after it started?", num_results=5)
[doc["question"] for doc in results]

['I just discovered the course. Can I still join?',
 'I missed the first homework - can I still get a certificate?',
 'Do we submit 2 projects, what does attempt 1 and 2 mean?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'I am using Azure OpenAI and I am still getting an error of Error code: 400 - {\'error\': {\'message\': "Missing required parameter: \'tools[0].function\'.", \'type\': \'invalid_request_error\', \'param\': \'tools[0].function\', \'code\': \'missing_required_parameter\'}}?']

In [4]:
from rag_helper import RAGBase
from openai import OpenAI

openai_client = OpenAI()

assistant = RAGBase(
    index=sqlite_index,
    llm_client=openai_client,
)

In [5]:
answer = assistant.rag("Can I still join the course after it started?")
print(answer)

Yes, you can still join after it has started. If you want a certificate, you need to submit your project while submissions are still open.


With minsearch (single process):

```python
Startup: fetch data -> parse -> index -> ready
Every restart: repeat all steps
```

With sqlitesearch (two processes):
```python
Ingestion (runs once): fetch data -> parse -> write to faq.db
Query (runs every time): open faq.db -> search -> ready
```